**Week 5 — Python สำหรับ Statistical Learning (pandas/numpy, Linear vs KNN, Test MSE)**

### สรุปเงื่อนไขการประมวลผลข้อมูล
- **การอ่านไฟล์:**
  - ข้อมูลลูกค้า: `cust = pd.read_csv('biz_customers.csv')` (1203 แถว)
  - ข้อมูลแคมเปญ: `camp = pd.read_csv('biz_campaigns.csv')` (240 แถว)
- **การนับดัชนี (0-based Indexing):** เริ่มนับจาก 0 ทั้งแถวและคอลัมน์ แถวคงลำดับเดิมตามไฟล์
- **การแบ่ง Train / Test:** แบ่งตามตำแหน่งแถว (`.iloc[]`) ไม่สุ่มแถว และไม่ใช้ `train_test_split`
- **โมเดลและพารามิเตอร์:** ใช้ค่าเริ่มต้นของ scikit-learn ทุกอย่าง (KNN ใช้ระยะทาง Euclidean และ uniform weights) ไม่ต้อง standardize ตัวทำนายยกเว้นที่โจทย์ระบุเฉพาะข้อ

### รายละเอียดไฟล์ข้อมูล

**ไฟล์ลูกค้า `biz_customers.csv`** (1203 แถว)
- `age`: อายุของลูกค้า (ปี)
- `income_k`: รายได้ต่อเดือน (พันบาท)
- `tenure_months`: ระยะเวลาที่เป็นลูกค้า (เดือน)
- `channel`: ช่องทางที่ซื้อบ่อย (`online` = ออนไลน์, `store` = หน้าร้าน)
- `loyalty_member`: สมาชิกสะสมแต้ม (1 = เป็น, 0 = ไม่เป็น)
- `web_visits`: จำนวนครั้งที่เข้าเว็บ/แอปใน 30 วันล่าสุด
- `email_opens`: จำนวนอีเมลโปรโมชันที่เปิดอ่าน (จาก 20 ฉบับ)
- `basket_size`: จำนวนสินค้าเฉลี่ยต่อออเดอร์ (ขนาดตะกร้า)
- `spend_k`: ยอดซื้อต่อเดือน (พันบาท)
- `orders_90d`: จำนวนออเดอร์ใน 90 วันล่าสุด

**ไฟล์แคมเปญ `biz_campaigns.csv`** (240 แถว)
- `tv`: งบโฆษณาทางโทรทัศน์ (พันบาท)
- `radio`: งบโฆษณาทางวิทยุ (พันบาท)
- `social`: งบโฆษณาทางโซเชียลมีเดีย (พันบาท)
- `email`: งบโฆษณาทางอีเมล (พันบาท)
- `season`: ไตรมาสของแคมเปญ (`Q1`–`Q4`)
- `sales`: ยอดขายของแคมเปญ (พันชิ้น)
- `total_budget`: งบโฆษณารวมทุกสื่อ (พันบาท)

**โจทย์ข้อ 5.1:** กรณีศึกษาทีมวิเคราะห์ลูกค้าของร้านค้าปลีกออนไลน์ต้องการประเมินว่าโมเดลเชิงเส้นทำนายยอดซื้อต่อเดือน (`spend_k`) ของลูกค้ารายใหม่ได้แม่นเพียงใด ทีมแบ่งข้อมูลดังนี้ train = 960 แถวแรกของ `biz_customers.csv` (แถวที่ 0–959), test = 243 แถวที่เหลือ (แถวที่ 960–1202) ; ตัวทำนาย $X$ = `income_k`, `tenure_months` และ `email_opens` (ตามลำดับนี้) และ $y$ = `spend_k` fit LinearRegression ด้วยชุด train แล้วนำไปทำนายชุด test ค่า Test MSE = ? (ทศนิยม 3 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# 1. อ่านข้อมูลลูกค้า
cust = pd.read_csv('biz_customers.csv')

# 2. แบ่ง train (แถว 0-959) และ test (แถว 960-1202)
train = cust.iloc[:960]
test = cust.iloc[960:]

features = ['income_k', 'tenure_months', 'email_opens']
X_train, y_train = train[features], train['spend_k']
X_test, y_test = test[features], test['spend_k']

# 3. Fit Linear Regression บน train และทำนาย test
model = LinearRegression().fit(X_train, y_train)
y_pred = model.predict(X_test)

# 4. คำนวณ Test MSE
test_mse = mean_squared_error(y_test, y_pred)
print(f"Test MSE: {test_mse:.3f}")
print(f"คำตอบข้อ 5.1: {test_mse:.3f}")


Test MSE: 1.949
คำตอบข้อ 5.1: 1.949


**โจทย์ข้อ 5.2:** ฝึกจากลูกค้าทั้ง 1203 คนใน `biz_customers.csv` หาค่าเฉลี่ยของ `basket_size` เฉพาะลูกค้าที่ `channel` = `'online'`, `loyalty_member` = 0 และ `tenure_months` > 54 (มากกว่าอย่างเคร่งครัด) พร้อมกันทั้งสามเงื่อนไข = ? (ทศนิยม 3 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd

# 1. อ่านข้อมูลลูกค้าทั้งหมด 1203 คน
cust = pd.read_csv('biz_customers.csv')

# 2. กรองข้อมูลตามเงื่อนไขทั้ง 3 ข้อพร้อมกัน
cond = (cust['channel'] == 'online') & (cust['loyalty_member'] == 0) & (cust['tenure_months'] > 54)
filtered_cust = cust[cond]
print(f"จำนวนลูกค้าที่ตรงตามเงื่อนไข: {len(filtered_cust)} คน")

# 3. หาค่าเฉลี่ยของ basket_size
ans_52 = filtered_cust['basket_size'].mean()
print(f"ค่าเฉลี่ย basket_size: {ans_52:.3f}")
print(f"คำตอบข้อ 5.2: {ans_52:.3f}")


จำนวนลูกค้าที่ตรงตามเงื่อนไข: 40 คน
ค่าเฉลี่ย basket_size: 3.070
คำตอบข้อ 5.2: 3.070


**โจทย์ข้อ 5.3:** ฝึกให้ $X$ เป็น array ขนาด $5 \times 3$ จากแถวที่ 34 ถึง 38 ของ `biz_campaigns.csv` (`camp.iloc[34:39]`) คอลัมน์ `tv`, `radio`, `social` ตามลำดับนี้ และสูตรทำนายยอดขาย $\hat{y} = 3.6 + 0.04 \times \text{tv} + 0.14 \times \text{radio} + 0.02 \times \text{social}$ (คำนวณทั้ง 5 แถวด้วย $X \beta + b_0$ ไม่ต้อง fit โมเดล) ค่าเฉลี่ยของค่าทำนาย $\hat{y}$ ทั้ง 5 แถว = ? (ทศนิยม 3 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd

# 1. อ่านข้อมูลแคมเปญแถวที่ 34 ถึง 38 (5 แถว)
camp = pd.read_csv('biz_campaigns.csv')
X = camp.iloc[34:39][['tv', 'radio', 'social']].values
print(f"ขนาดของ X: {X.shape}")

# 2. คำนวณค่าทำนาย y_hat = b0 + X @ beta
b0 = 3.6
beta = np.array([0.04, 0.14, 0.02])
y_hat = b0 + X @ beta
print(f"ค่าทำนาย y_hat ทั้ง 5 แถว: {y_hat}")

# 3. หาค่าเฉลี่ยของค่าทำนายทั้ง 5 แถว
ans_53 = np.mean(y_hat)
print(f"ค่าเฉลี่ยของ y_hat: {ans_53:.3f}")
print(f"คำตอบข้อ 5.3: {ans_53:.3f}")


ขนาดของ X: (5, 3)
ค่าทำนาย y_hat ทั้ง 5 แถว: [15.946 13.718 15.692 10.23   8.684]
ค่าเฉลี่ยของ y_hat: 12.854
คำตอบข้อ 5.3: 12.854


**โจทย์ข้อ 5.4:** ฝึกเฉพาะแคมเปญใน `biz_campaigns.csv` ที่ `season` = `'Q1'` หาสัมประสิทธิ์สหสัมพันธ์ Pearson ระหว่าง `tv` กับ `radio` (ค่าเดียวกับในตาราง `corr()` ของ pandas ที่คำนวณจากแถวเฉพาะไตรมาสนี้) = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd

# 1. อ่านข้อมูลเฉพาะไตรมาส Q1
camp = pd.read_csv('biz_campaigns.csv')
camp_q1 = camp[camp['season'] == 'Q1']
print(f"จำนวนแคมเปญใน Q1: {len(camp_q1)} แถว")

# 2. คำนวณ Pearson correlation ระหว่าง tv กับ radio
corr_matrix = camp_q1[['tv', 'radio']].corr()
print("Correlation matrix:")
print(corr_matrix)

ans_54 = corr_matrix.loc['tv', 'radio']
print(f"คำตอบข้อ 5.4: {ans_54:.4f}")


จำนวนแคมเปญใน Q1: 60 แถว
Correlation matrix:
             tv     radio
tv     1.000000  0.187747
radio  0.187747  1.000000
คำตอบข้อ 5.4: 0.1877


**โจทย์ข้อ 5.5:** กรณีศึกษาทีมการตลาดดิจิทัลของเอเจนซี่โฆษณาต้องการเลือกจำนวนแคมเปญเก่าที่ใกล้เคียงที่สุด ($K$) ที่เหมาะสมสำหรับทำนายยอดขายของแคมเปญ (`sales`) จากงบโฆษณา `tv` และ `social` (ตามลำดับนี้ งบหน่วยพันบาท ไม่ต้อง standardize) โดยใช้ไฟล์ `biz_campaigns.csv` และ $y$ = `sales` ; train = 192 แถวแรกของ `biz_campaigns.csv` (แถวที่ 0–191), test = 48 แถวที่เหลือ (แถวที่ 192–239) ทีมลอง `KNeighborsRegressor` ที่ $K$ (`n_neighbors`) = $1, 2, \dots, 60$ ทีละค่า fit ด้วยชุด train แล้วคำนวณ Test MSE ของแต่ละ $K$ ค่า $K$ ที่ทำให้ **Test MSE ต่ำที่สุด** = ? (จำนวนเต็ม)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error

# 1. อ่านข้อมูลและแบ่ง train (192 แถวแรก) และ test (48 แถวที่เหลือ)
camp = pd.read_csv('biz_campaigns.csv')
train = camp.iloc[:192]
test = camp.iloc[192:]

X_train = train[['tv', 'social']]
y_train = train['sales']
X_test = test[['tv', 'social']]
y_test = test['sales']

# 2. ทดลอง K = 1 ถึง 60 และบันทึก Test MSE
best_k = None
min_mse = float('inf')

for k in range(1, 61):
    knn = KNeighborsRegressor(n_neighbors=k).fit(X_train, y_train)
    y_pred = knn.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    if mse < min_mse:
        min_mse = mse
        best_k = k

print(f"ค่า K ที่ทำให้ Test MSE ต่ำที่สุด: K = {best_k} (Test MSE = {min_mse:.4f})")
print(f"คำตอบข้อ 5.5: {best_k}")


ค่า K ที่ทำให้ Test MSE ต่ำที่สุด: K = 20 (Test MSE = 29.6386)
คำตอบข้อ 5.5: 20


**โจทย์ข้อ 5.6:** ฝึกแบ่งข้อมูลดังนี้ train = 960 แถวแรกของ `biz_customers.csv` (แถวที่ 0–959), test = 243 แถวที่เหลือ (แถวที่ 960–1202) ; ตัวทำนาย $X$ = `loyalty_member` และ `basket_size` (ตามลำดับนี้) และ $y$ = `orders_90d` fit LinearRegression ด้วยชุด train แล้วหาค่า $R^2$ บนชุด test (ค่าเดียวกับ score ของโมเดลบนชุด test ; ค่าเฉลี่ยที่ใช้ใน $R^2$ คือค่าเฉลี่ยของ $y$ ในชุด test) = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

# 1. อ่านข้อมูลลูกค้าและแบ่ง train / test
cust = pd.read_csv('biz_customers.csv')
train = cust.iloc[:960]
test = cust.iloc[960:]

features = ['loyalty_member', 'basket_size']
X_train, y_train = train[features], train['orders_90d']
X_test, y_test = test[features], test['orders_90d']

# 2. Fit Linear Regression บน train
model = LinearRegression().fit(X_train, y_train)

# 3. หาค่า R^2 บนชุด test (ผ่าน score method)
r2_test = model.score(X_test, y_test)
print(f"R^2 บนชุด test: {r2_test:.4f}")
print(f"คำตอบข้อ 5.6: {r2_test:.4f}")


R^2 บนชุด test: 0.1181
คำตอบข้อ 5.6: 0.1181


**โจทย์ข้อ 5.7:** กรณีศึกษาฝ่ายวางแผนสื่อของแบรนด์เครื่องสำอางกำลังพิจารณาแคมเปญใหม่ที่มีงบโฆษณาทีวี (`tv`) = 204.4, งบโฆษณาวิทยุ (`radio`) = 41.7 (พันบาท) และต้องการค่าประมาณยอดขายของแคมเปญ (`sales`, พันชิ้น) จากโมเดลเชิงเส้น ให้ fit LinearRegression ด้วยตัวทำนาย `tv` และ `radio` และ $y$ = `sales` โดยใช้ 192 แถวแรกของ `biz_campaigns.csv` (แถวที่ 0–191) เป็นข้อมูลสร้างโมเดล แล้วทำนายยอดขายของแคมเปญใหม่นี้ = ? (ทศนิยม 3 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

# 1. ใช้ 192 แถวแรกเป็นชุดสร้างโมเดล
camp = pd.read_csv('biz_campaigns.csv')
train = camp.iloc[:192]
X_train = train[['tv', 'radio']]
y_train = train['sales']

# 2. Fit Linear Regression
model = LinearRegression().fit(X_train, y_train)
print(f"Intercept (b0): {model.intercept_:.4f}")
print(f"Coefficients: tv={model.coef_[0]:.4f}, radio={model.coef_[1]:.4f}")

# 3. ทำนายยอดขายของแคมเปญใหม่: tv = 204.4, radio = 41.7
new_campaign = pd.DataFrame([{'tv': 204.4, 'radio': 41.7}])
pred_sales = model.predict(new_campaign)[0]
print(f"ยอดขายที่ทำนายได้: {pred_sales:.3f}")
print(f"คำตอบข้อ 5.7: {pred_sales:.3f}")


Intercept (b0): 3.0118
Coefficients: tv=0.0456, radio=0.1470
ยอดขายที่ทำนายได้: 28.463
คำตอบข้อ 5.7: 28.463


**โจทย์ข้อ 5.8:** กรณีศึกษาผู้บริหารธุรกิจค้าปลีกถามว่าการสร้างโมเดลเชิงเส้นคุ้มค่าหรือไม่เมื่อเทียบกับโมเดลพื้นฐาน (baseline) ที่ทำนายยอดซื้อต่อเดือน (`spend_k`) ของลูกค้าทุกคนด้วยค่าเดียว คือค่าเฉลี่ยของ `spend_k` ในชุด train แบ่งข้อมูลดังนี้ train = 960 แถวแรกของ `biz_customers.csv` (แถวที่ 0–959), test = 243 แถวที่เหลือ (แถวที่ 960–1202) ; โมเดลเชิงเส้นคือ LinearRegression ที่ใช้ $X$ = `income_k`, `web_visits` และ `basket_size` (ตามลำดับนี้) fit ด้วยชุด train ให้หาอัตราส่วน **(Test MSE ของโมเดลเชิงเส้น) ÷ (Test MSE ของ baseline)** = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# 1. อ่านข้อมูลลูกค้าและแบ่ง train / test
cust = pd.read_csv('biz_customers.csv')
train = cust.iloc[:960]
test = cust.iloc[960:]

features = ['income_k', 'web_visits', 'basket_size']
X_train, y_train = train[features], train['spend_k']
X_test, y_test = test[features], test['spend_k']

# 2. Test MSE ของโมเดลเชิงเส้น
lr = LinearRegression().fit(X_train, y_train)
pred_lr = lr.predict(X_test)
mse_lr = mean_squared_error(y_test, pred_lr)

# 3. Test MSE ของโมเดลพื้นฐาน (baseline ทำนายด้วยค่าเฉลี่ยของ y_train)
baseline_pred = np.full_like(y_test, y_train.mean())
mse_baseline = mean_squared_error(y_test, baseline_pred)

ratio = mse_lr / mse_baseline
print(f"MSE โมเดลเชิงเส้น: {mse_lr:.4f}")
print(f"MSE โมเดล Baseline: {mse_baseline:.4f}")
print(f"อัตราส่วน (MSE Linear / MSE Baseline): {ratio:.4f}")
print(f"คำตอบข้อ 5.8: {ratio:.4f}")


MSE โมเดลเชิงเส้น: 1.7618
MSE โมเดล Baseline: 2.3538
อัตราส่วน (MSE Linear / MSE Baseline): 0.7485
คำตอบข้อ 5.8: 0.7485


**โจทย์ข้อ 5.9:** กรณีศึกษาผู้จัดการแบรนด์อยากรู้ว่าแคมเปญหมายเลขแถวที่ 211 ของ `biz_campaigns.csv` (อยู่ในชุด test) จะถูกโมเดล "ดูจากแคมเปญเก่าที่ใช้งบใกล้เคียงกันที่สุด" ทำนายยอดขายไว้เท่าใด โมเดลคือ `KNeighborsRegressor(n_neighbors=9)` ที่ fit ด้วยชุด train (192 แถวแรก คือแถวที่ 0–191) ใช้ $X$ = `tv` และ `radio` (ตามลำดับนี้ งบหน่วยพันบาท) และ $y$ = `sales` ค่าทำนายยอดขาย (`sales`) ของแคมเปญแถวที่ 211 = ? (ทศนิยม 3 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsRegressor

# 1. ใช้ 192 แถวแรกสร้างโมเดล KNN (k=9)
camp = pd.read_csv('biz_campaigns.csv')
train = camp.iloc[:192]
X_train = train[['tv', 'radio']]
y_train = train['sales']

knn = KNeighborsRegressor(n_neighbors=9).fit(X_train, y_train)

# 2. ดึงข้อมูลแคมเปญแถวที่ 211 (0-indexed)
row_211 = camp.iloc[211:212][['tv', 'radio']]
print(f"ค่างบโฆษณาแถว 211:\n{row_211}")

# 3. ทำนายยอดขาย
pred_sales_211 = knn.predict(row_211)[0]
print(f"ยอดขายที่ทำนายได้: {pred_sales_211:.3f}")
print(f"คำตอบข้อ 5.9: {pred_sales_211:.3f}")


ค่างบโฆษณาแถว 211:
        tv  radio
211  158.3   24.3
ยอดขายที่ทำนายได้: 15.442
คำตอบข้อ 5.9: 15.442


**โจทย์ข้อ 5.10:** กรณีศึกษาเอเจนซี่โฆษณาต้องการเปรียบเทียบ `LinearRegression` กับ `KNeighborsRegressor(n_neighbors=7)` ในรายแคมเปญ ทั้งสองโมเดลใช้ $X$ = `tv`, `radio` และ `email` (ตามลำดับนี้) และ $y$ = `sales` fit ด้วยชุด train แล้วทำนายชุด test โดย train = 192 แถวแรกของ `biz_campaigns.csv` (แถวที่ 0–191), test = 48 แถวที่เหลือ (แถวที่ 192–239) นับจำนวนแคมเปญในชุด test ที่ค่าทำนายของ KNN มีความคลาดเคลื่อนสัมบูรณ์ $|\text{ค่าจริง} - \text{ค่าทำนาย}|$ **น้อยกว่าของ Linear Regression (น้อยกว่าอย่างเคร่งครัด)** = ? (จำนวนเต็ม)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor

# 1. อ่านข้อมูลและแบ่ง train (192 แถว) และ test (48 แถว)
camp = pd.read_csv('biz_campaigns.csv')
train = camp.iloc[:192]
test = camp.iloc[192:]

features = ['tv', 'radio', 'email']
X_train, y_train = train[features], train['sales']
X_test, y_test = test[features], test['sales']

# 2. Fit และทำนายด้วย Linear Regression
lr = LinearRegression().fit(X_train, y_train)
pred_lr = lr.predict(X_test)

# 3. Fit และทำนายด้วย KNN (k=7)
knn = KNeighborsRegressor(n_neighbors=7).fit(X_train, y_train)
pred_knn = knn.predict(X_test)

# 4. คำนวณความคลาดเคลื่อนสัมบูรณ์ |y_actual - y_pred|
err_lr = np.abs(y_test.values - pred_lr)
err_knn = np.abs(y_test.values - pred_knn)

# 5. นับจำนวนแคมเปญที่ err_knn < err_lr อย่างเคร่งครัด
count_better = int(np.sum(err_knn < err_lr))
print(f"จำนวนแคมเปญที่ KNN คลาดเคลื่อนน้อยกว่า Linear Regression: {count_better} แคมเปญ (จาก 48 แคมเปญ)")
print(f"คำตอบข้อ 5.10: {count_better}")


จำนวนแคมเปญที่ KNN คลาดเคลื่อนน้อยกว่า Linear Regression: 17 แคมเปญ (จาก 48 แคมเปญ)
คำตอบข้อ 5.10: 17


## สรุปทฤษฎีและแนวคิดหลักประจำ Week 5

---

### 1. กรอบแนวคิด Statistical Learning: Parametric vs Non-parametric

เป้าหมายของ Statistical Learning คือการประมาณฟังก์ชัน $f$ จากข้อมูลตัวอย่าง $(X, Y)$ ตามแบบจำลอง:
$$Y = f(X) + \epsilon, \quad \mathbb{E}[\epsilon] = 0, \quad \text{Var}(\epsilon) = \sigma^2$$

- **Parametric Methods (เช่น Linear Regression):**
  - มีสมมติฐานเกี่ยวกับรูปทรงฟังก์ชันไว้ล่วงหน้า เช่น $f(X) = \beta_0 + \beta_1 X_1 + \dots + \beta_p X_p$
  - จุดเด่น: ตีความหมายง่าย (interpretable) ใช้ข้อมูลจำนวนน้อยในการประมาณพารามิเตอร์ และไม่ค่อย overfit ได้ง่าย
  - ข้อจำกัด: หากความสัมพันธ์ที่แท้จริงไม่เป็นเส้นตรง (non-linear) โมเดลจะมี Bias สูง

- **Non-parametric Methods (เช่น K-Nearest Neighbors):**
  - ไม่มีสมมติฐานเรื่องรูปทรงทางคณิตศาสตร์ของ $f(X)$ ล่วงหน้า
  - ประมาณค่าโดยตรงจากความใกล้เคียงกันของข้อมูลในพื้นที่ท้องถิ่น (Local averaging)
  - จุดเด่น: ยืดหยุ่นสูง จับรูปแบบซับซ้อนและ non-linear ได้ดี
  - ข้อจำกัด: ตีความยาก และประสิทธิภาพลดลงเมื่อจำนวนตัวแปร $p$ เพิ่มขึ้น (Curse of Dimensionality)

---

### 2. K-Nearest Neighbors Regression (KNN)

สำหรับจุดข้อมูลทดสอบ $x_0$ อัลกอริทึม KNN จะค้นหา $K$ จุดในชุดฝึกหัดที่ใกล้ $x_0$ ที่สุด (เรียกว่า $\mathcal{N}_0$) ตามระยะทาง Euclidean:
$$d(x_0, x_i) = \|x_0 - x_i\|_2 = \sqrt{\sum_{j=1}^p (x_{0j} - x_{ij})^2}$$

แล้วประมาณค่าผลลัพธ์ด้วยค่าเฉลี่ยของเพื่อนบ้าน $K$ ตัว:
$$\hat{f}(x_0) = \frac{1}{K} \sum_{i \in \mathcal{N}_0} y_i$$

- **บทบาทของ Hyperparameter $K$:**
  - $K$ น้อย (เช่น $K=1$): โมเดลยืดหยุ่นสูงมาก (High Variance, Low Bias) ไวต่อ noise เสี่ยงต่อการ Overfitting
  - $K$ มาก (เช่น $K=60$): โมเดลเรียบขึ้น (High Bias, Low Variance) ค่าทำนายจะลู่เข้าหาค่าเฉลี่ยของทั้งชุดข้อมูล
- การเลือก $K$ ที่เหมาะสมทำได้โดยการหาค่า $K$ ที่ให้ **Test MSE ต่ำที่สุด** (หรือใช้ Cross-Validation)

---

### 3. การประเมินผลโมเดลและตัวชี้วัด (Evaluation Metrics)

- **Mean Squared Error (MSE):**
  $$\text{MSE} = \frac{1}{n} \sum_{i=1}^n (y_i - \hat{y}_i)^2$$
  - **Train MSE:** วัดความคลาดเคลื่อนบนข้อมูลที่ใช้สอน (ยิ่งโมเดลซับซ้อน Train MSE ยิ่งลดลง)
  - **Test MSE:** วัดความสามารถในการพยากรณ์ข้อมูลใหม่ที่ไม่เคยเห็น (Generalization error) ซึ่งเป็นเกณฑ์ตัดสินประสิทธิภาพที่แท้จริง

- **สัมประสิทธิ์การตัดสินใจ ($R^2$ Score):**
  $$R^2 = 1 - \frac{\sum_{i=1}^n (y_i - \hat{y}_i)^2}{\sum_{i=1}^n (y_i - \bar{y})^2} = 1 - \frac{\text{MSE}(\text{model})}{\text{Var}(y)}$$
  สัดส่วนความแปรปรวนของ $y$ ที่โมเดลสามารถอธิบายได้ (ค่าสูงสุดคือ 1.0; บนชุด test อาจติดลบได้หากโมเดลแย่กว่าการทายด้วยค่าเฉลี่ย)

---

### 4. Baseline Model และ Relative Performance

- **โมเดลพื้นฐาน (Baseline Model):** ทำนายทุกจุดข้อมูลด้วยค่าคงที่ค่าเดียวคือ $\bar{y}_{\text{train}}$
- **อัตราส่วน Relative MSE:**
  $$\text{Ratio} = \frac{\text{MSE}_{\text{model}}}{\text{MSE}_{\text{baseline}}}$$
  - ค่า $\text{Ratio} < 1.0$: แสดงว่าตัวทำนาย $X$ มีข้อมูลที่มีประโยชน์และช่วยลดความคลาดเคลื่อนได้จริงเมื่อเทียบกับการเดาด้วยค่าเฉลี่ย
  - ค่า $\text{Ratio} = 0.7485$ หมายถึงโมเดลเชิงเส้นช่วยลดความคลาดเคลื่อนลงได้ประมาณ $25.15\%$ เมื่อเทียบกับโมเดลพื้นฐาน